# WS-03 — Gán nhãn yếu trên Google Colab

Notebook này chạy trên **Google Colab** cho bước 3 của quy trình Weak Supervision:

1. Kết nối Drive và lấy đúng branch `WS`.
2. Cài phụ thuộc Colab tối thiểu của RAD.
3. Đọc `configs/ws.yaml` và xác thực các đường dẫn SkinCAP đã được kiểm chứng.
4. Chỉ đọc và kiểm tra toàn bộ artifact đã chốt của WS_02 trên Drive.
5. Tạo bản nháp alias bệnh để Trí duyệt, rồi chạy NLI trực tiếp trong notebook và lưu checkpoint riêng ngay lập tức.
6. Chạy ConceptCLIP trực tiếp trong một ô độc lập, lưu checkpoint ảnh riêng ngay lập tức.
7. Hợp nhất các checkpoint, tạo dry-run LLM, xuất `labels.npz` và khảo sát phân bố nhãn trong `weak_labels/v1/`.

> **Ranh giới WS_02 / WS_03:** `propositions_review.csv`, `propositions_final.json`, `summary_per_disease.csv` và `meta.json` trong `WeakSup/propositions/v1/` là đầu vào **read-only** của WS_03. Notebook này không export, tạo lại, sửa hoặc ghi đè bất kỳ file WS_02 nào.
>
> **Checkpoint bắt buộc:** NLI và ConceptCLIP chạy ở các ô riêng, mỗi phần lưu checkpoint vào `weak_labels/v1/checkpoints/` trước khi bước sau bắt đầu. Không chạy NLI qua subprocess rồi nối thẳng sang ConceptCLIP.
>
> **Bảo vệ LLM:** notebook chỉ tạo `llm_dry_run_report.csv`; không gọi LLM thật hoặc đọc API key.
>
> Không lưu Hugging Face token, OpenAI API key hoặc Anthropic API key vào notebook hay YAML. Chỉ nhập chúng qua Colab Secrets / biến môi trường khi runner bước 3 đã sẵn sàng.

## 1. Thiết lập môi trường Google Colab

Kiểm tra phiên bản Python, GPU khả dụng và thư mục làm việc hiện tại. Chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU tương đương) trước khi chạy các phần liên quan đến mô hình.

In [ ]:
import os
import platform
import sys

try:
    import torch
except ImportError as error:
    raise RuntimeError("Notebook này phải được chạy trong Google Colab có PyTorch.") from error

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Working directory: {os.getcwd()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA runtime: {torch.version.cuda}")
else:
    print("⚠️ Chưa có GPU. Bước gán nhãn mô hình nên chạy với GPU Colab.")

## 2. Cài đặt và nhập các thư viện

Cài bộ phụ thuộc tập trung cho SkinCAP trên Colab. Không dùng `requirements.txt` đầy đủ vì nó chứa nhiều gói phục vụ môi trường/nhánh khác, một số gói cần biên dịch và không phù hợp với Colab.

In [ ]:
# Các gói cơ bản cần cho các ô kiểm tra và đọc cấu hình.
# Các phụ thuộc SkinCAP/RAD sẽ được cài sau khi clone repository.
!pip -q install --no-cache-dir "PyYAML>=6.0" pandas numpy tqdm

import json
import subprocess
from pathlib import Path

import pandas as pd
import yaml

print("Đã import: pandas và PyYAML.")

## 3. Kết nối Google Drive và lấy branch `WS`

Drive cung cấp dữ liệu SkinCAP và nơi lưu artifact phiên bản. Repository được đặt tại `/content/RAD-colab`; ô này chỉ checkout branch `WS`, không tự tạo hay đổi các đường dẫn dữ liệu đã xác nhận.

In [ ]:
try:
    from google.colab import drive
except ImportError as error:
    raise RuntimeError("Hãy chạy notebook này trong Google Colab.") from error

drive.mount("/content/drive", force_remount=False)

REPO_URL = "https://github.com/trisle0805/RAD-colab.git"
REPO_DIR = Path("/content/RAD-colab")
BRANCH = "WS"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

current_branch = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True
).strip()
assert current_branch == BRANCH, f"Expected branch {BRANCH!r}, got {current_branch!r}"
print(f"Repository: {REPO_DIR}")
print(f"Branch: {current_branch}")

In [ ]:
# The repository has now been cloned, so install its focused Colab dependency set.
requirements_file = REPO_DIR / "requirements" / "colab-skin.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Missing Colab requirements file: {requirements_file}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "-r", str(requirements_file)],
    check=True,
)
print(f"Đã cài phụ thuộc từ: {requirements_file}")

## 4. Khai báo đường dẫn dữ liệu và kết quả

Tất cả các đường dẫn được lấy từ `configs/ws.yaml`. Artifact WS_02 trong `WeakSup/propositions/v1/` là **đầu vào chỉ đọc**. Notebook không sao chép, di chuyển, xuất lại hay chỉnh sửa file SkinCAP hoặc artifact WS_02.

In [ ]:
CONFIG_PATH = REPO_DIR / "configs" / "ws.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing WS config: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

# Repository paths are intentionally asserted rather than silently changed.
assert Path(cfg["repo_dir"]) == REPO_DIR, (
    f"configs/ws.yaml repo_dir must be {REPO_DIR}, got {cfg['repo_dir']}"
)

PATHS = {
    "train_csv": Path(cfg["skincap"]["train_csv"]),
    "val_csv": Path(cfg["skincap"]["val_csv"]),
    "test_csv": Path(cfg["skincap"]["test_csv"]),
    "image_root": Path(cfg["skincap"]["image_root"]),
    # Read-only artifacts produced by WS_02 on Drive.
    "propositions_dir": Path(cfg["propositions_dir"]),
    "propositions_review": Path(cfg["propositions_review"]),
    "propositions_final": Path(cfg["propositions_final"]),
    "propositions_summary": Path(cfg["propositions_summary"]),
    "propositions_meta": Path(cfg["propositions_meta"]),
    # Future Stage-03 inputs/outputs; their existence is checked separately.
    "teacher_text_model": Path(cfg["teacher_text_model"]),
    "weak_labels_dir": Path(cfg["weak_labels_dir"]),
}

print("Cấu hình đã nạp từ:", CONFIG_PATH)
for name, path in PATHS.items():
    print(f"- {name}: {path}")

In [ ]:
def require_file(label: str, path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"{label} không tồn tại: {path}")
    return path


def require_directory(label: str, path: Path) -> Path:
    if not path.is_dir():
        raise NotADirectoryError(f"{label} không tồn tại hoặc không phải thư mục: {path}")
    return path


for split in ("train_csv", "val_csv", "test_csv"):
    require_file(split, PATHS[split])
require_directory("image_root", PATHS["image_root"])

# WS_03 requires the complete, already-approved WS_02 version on Drive.
require_directory("propositions_dir", PATHS["propositions_dir"])
for artifact in (
    "propositions_review",
    "propositions_final",
    "propositions_summary",
    "propositions_meta",
):
    require_file(artifact, PATHS[artifact])

# Check split schema and case counts without copying data.
split_frames = {}
for split in ("train", "val", "test"):
    csv_path = PATHS[f"{split}_csv"]
    frame = pd.read_csv(csv_path)
    if frame.empty:
        raise ValueError(f"{split} split is empty: {csv_path}")
    split_frames[split] = frame
    print(f"{split:>5}: {len(frame):,} rows | columns: {list(frame.columns)}")

image_count = sum(1 for path in PATHS["image_root"].iterdir() if path.is_file())
print(f"Images directly in image_root: {image_count:,}")
print("WS_02 artifacts: present and read-only.")

## 5. Đọc artifact đã chốt của WS_02

Bước 3 chỉ đọc `propositions_final.json` để lấy danh sách proposition, hypothesis và cờ `image_applicable`. `propositions_review.csv`, `summary_per_disease.csv` và `meta.json` được kiểm tra để bảo đảm bộ artifact WS_02 đầy đủ và truy vết được. Notebook không gọi `export_final_version` và không ghi vào thư mục `WeakSup/propositions/v1/`.

In [ ]:
propositions_dir = PATHS["propositions_dir"]
propositions_final = PATHS["propositions_final"]

# Read-only: WS_02 has already produced and approved this version on Drive.
with propositions_final.open("r", encoding="utf-8") as handle:
    propositions = json.load(handle)

if not isinstance(propositions, list) or not propositions:
    raise ValueError(f"Invalid or empty propositions file: {propositions_final}")

with PATHS["propositions_meta"].open("r", encoding="utf-8") as handle:
    propositions_meta = json.load(handle)

summary = pd.read_csv(PATHS["propositions_summary"])
if summary.empty:
    raise ValueError(f"Empty WS_02 summary: {PATHS['propositions_summary']}")

print(f"WS_02 directory (read-only): {propositions_dir}")
print(f"Observable propositions: {len(propositions):,}")
print(f"Diseases in WS_02 summary: {len(summary):,}")
print(f"WS_02 stage: {propositions_meta.get('stage', 'unknown')}")
print("First proposition:", json.dumps(propositions[0], ensure_ascii=False, indent=2))

## 5.1 Kiểm tra điều kiện trước khi gán nhãn yếu

Checkpoint Bio_ClinicalBERT từ bước 1 là điều kiện bắt buộc. Các ô sau gọi NLI và ConceptCLIP trực tiếp trong notebook, để lỗi đầy đủ hiển thị ngay trong output của Colab.

In [ ]:
teacher_ready = PATHS["teacher_text_model"].is_dir()

print(f"Text teacher checkpoint: {'READY' if teacher_ready else 'MISSING'}")
print(f"  {PATHS['teacher_text_model']}")
print(f"ConceptCLIP model ID (config): {cfg['image_labeler']['model_id']}")

if not teacher_ready:
    print("\nACTION REQUIRED: Kiểm tra checkpoint WS_01 trên Drive.")
print("\nCác ô phía dưới không gọi API LLM thật.")

## 6. Chuẩn bị alias và hàm dùng chung

NLI chỉ đọc caption đã che alias bệnh. Ô này tạo `disease_alias_v1.csv` nếu chưa có, sau đó nạp các helper thuần dữ liệu. Không chạm vào WS_02.

In [ ]:
import hashlib

import numpy as np
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_DIR))
from proposition.weak.labels import (
    ABSENT,
    LLM_GROUP_CODES,
    LLM_NOT_SENT,
    PRESENT,
    UNKNOWN,
    WeakLabels,
    load_propositions,
    mask_disease_aliases,
    merge_labels,
    read_alias_csv,
    split_sentences,
    suggested_alias_rows,
    write_alias_csv,
)
from proposition.weak.reporting import write_survey

OUTPUT_DIR = PATHS["weak_labels_dir"]
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
ALIAS_CSV = OUTPUT_DIR / "disease_alias_v1.csv"
NLI_CHECKPOINT = CHECKPOINT_DIR / "nli_checkpoint.npz"
IMAGE_CHECKPOINT = CHECKPOINT_DIR / "conceptclip_checkpoint.npz"

if not ALIAS_CSV.exists():
    disease_columns = list(split_frames["train"].columns[2:])
    write_alias_csv(ALIAS_CSV, suggested_alias_rows(disease_columns))

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_cases() -> tuple[list[dict[str, str]], list[str]]:
    cases: list[dict[str, str]] = []
    disease_columns: list[str] = []
    for split in ("train", "val", "test"):
        frame = pd.read_csv(PATHS[f"{split}_csv"], dtype=str).fillna("")
        case_column, caption_column = frame.columns[:2]
        if split == "train":
            disease_columns = list(frame.columns[2:])
        for _, item in frame.iterrows():
            positives = [name for name in disease_columns if name in item and str(item[name]) in {"1", "1.0", "True", "true"}]
            cases.append({
                "case_id": str(item[case_column]),
                "split": split,
                "caption": str(item[caption_column]),
                "disease": positives[0] if len(positives) == 1 else "",
            })
    if len({case["case_id"] for case in cases}) != len(cases):
        raise ValueError("case_id phải là duy nhất trên train/val/test.")
    return cases, disease_columns

PROPOSITIONS = load_propositions(PATHS["propositions_final"], PATHS["propositions_review"])
CASES, _ = load_cases()
ALIASES = read_alias_csv(ALIAS_CSV)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Alias CSV:", ALIAS_CSV)
display(pd.read_csv(ALIAS_CSV).head(20))
print(f"Cases: {len(CASES):,}; propositions: {len(PROPOSITIONS):,}")
print("NLI checkpoint:", NLI_CHECKPOINT)
print("ConceptCLIP checkpoint:", IMAGE_CHECKPOINT)
print("Alias CSV:", ALIAS_CSV)
display(pd.read_csv(ALIAS_CSV).head(20))
print("Duyệt/chỉnh CSV trên Drive trước khi chạy ô tiếp theo.")

## 7. Chạy NLI và lưu checkpoint ngay lập tức

Ô này thực hiện BioClinicalBERT NLI trực tiếp trong kernel Colab. Khi NLI hoàn tất, nó **lưu ngay** `checkpoints/nli_checkpoint.npz`, `sentences_masked.json` và `llm_dry_run_report.csv` trước khi bất kỳ code ConceptCLIP nào chạy. Nếu ô sau bị lỗi, không chạy lại ô này: checkpoint NLI vẫn còn trên Drive.

In [ ]:
if not teacher_ready:
    raise RuntimeError("Thiếu checkpoint BioClinicalBERT; xem lại phần 5.1.")

# Resume-safe: checkpoint hợp lệ được nạp lại, NLI không chạy lại.
if NLI_CHECKPOINT.is_file():
    with np.load(NLI_CHECKPOINT, allow_pickle=False) as archive:
        required = {"case_id", "split", "proposition_id", "label_text_nli", "p_entail", "p_contra", "best_sentence_idx"}
        missing = required - set(archive.files)
        if missing:
            raise ValueError(f"NLI checkpoint thiếu arrays: {sorted(missing)}")
        if not np.array_equal(archive["case_id"], np.asarray([case["case_id"] for case in CASES])):
            raise ValueError("NLI checkpoint không khớp thứ tự case hiện tại.")
        if not np.array_equal(archive["proposition_id"], np.asarray([prop.proposition_id for prop in PROPOSITIONS])):
            raise ValueError("NLI checkpoint không khớp propositions WS_02 hiện tại.")
        nli_arrays = {key: archive[key] for key in required}
    print("Đã nạp checkpoint NLI, không chạy lại:", NLI_CHECKPOINT)
else:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    model_dir = PATHS["teacher_text_model"]
    label_map_path = model_dir.parent / "label_map.json"
    expected_label_map = {"entailment": 0, "neutral": 1, "contradiction": 2}
    if json.loads(label_map_path.read_text(encoding="utf-8")) != expected_label_map:
        raise ValueError("Thứ tự nhãn BioClinicalBERT không đúng entailment/neutral/contradiction.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Cần GPU Colab trước khi chạy NLI toàn bộ.")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()
    n_cases, n_props = len(CASES), len(PROPOSITIONS)
    label_text_nli = np.full((n_cases, n_props), UNKNOWN, dtype=np.int8)
    p_entail = np.zeros((n_cases, n_props), dtype=np.float16)
    p_contra = np.zeros((n_cases, n_props), dtype=np.float16)
    best_sentence_idx = np.full((n_cases, n_props), -1, dtype=np.int16)
    sentences_masked: dict[str, list[str]] = {}
    options = cfg["text_labeler"]
    batch_size = int(options["batch_size"])

    for row, case in enumerate(tqdm(CASES, desc="WS_03 NLI")):
        sentences = split_sentences(mask_disease_aliases(case["caption"], ALIASES))
        sentences_masked[case["case_id"]] = sentences
        if not sentences:
            continue
        tasks = [
            (column, hypothesis_index, sentence_index, sentence, hypothesis)
            for column, proposition in enumerate(PROPOSITIONS)
            for hypothesis_index, hypothesis in enumerate(proposition.hypotheses)
            for sentence_index, sentence in enumerate(sentences)
        ]
        entailment = [[0.0] * len(prop.hypotheses) for prop in PROPOSITIONS]
        contradiction = [[0.0] * len(prop.hypotheses) for prop in PROPOSITIONS]
        best = [[-1] * len(prop.hypotheses) for prop in PROPOSITIONS]
        for start in range(0, len(tasks), batch_size):
            batch = tasks[start:start + batch_size]
            encoded = tokenizer(
                [task[3] for task in batch], [task[4] for task in batch],
                truncation=True, max_length=int(options["max_length"]),
                padding=True, return_tensors="pt",
            )
            encoded = {key: value.to(device) for key, value in encoded.items()}
            with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=bool(options.get("fp16", False))):
                probabilities = torch.softmax(model(**encoded).logits.float(), dim=-1).cpu().numpy()
            for task, probability in zip(batch, probabilities):
                column, hypothesis_index, sentence_index = task[:3]
                if float(probability[0]) > entailment[column][hypothesis_index]:
                    entailment[column][hypothesis_index] = float(probability[0])
                    best[column][hypothesis_index] = sentence_index
                contradiction[column][hypothesis_index] = max(contradiction[column][hypothesis_index], float(probability[2]))
        for column, proposition in enumerate(PROPOSITIONS):
            p_entail[row, column] = max(entailment[column])
            p_contra[row, column] = max(contradiction[column])
            winner = int(np.argmax(entailment[column]))
            best_sentence_idx[row, column] = best[column][winner]
            if any(value >= float(options["entailment_threshold"]) for value in entailment[column]):
                label_text_nli[row, column] = PRESENT
            elif all(value >= float(options["contradiction_threshold"]) for value in contradiction[column]):
                label_text_nli[row, column] = ABSENT

    nli_arrays = {
        "case_id": np.asarray([case["case_id"] for case in CASES]),
        "split": np.asarray([case["split"] for case in CASES]),
        "proposition_id": np.asarray([prop.proposition_id for prop in PROPOSITIONS]),
        "label_text_nli": label_text_nli,
        "p_entail": p_entail,
        "p_contra": p_contra,
        "best_sentence_idx": best_sentence_idx,
    }
    np.savez_compressed(NLI_CHECKPOINT, **nli_arrays)
    (OUTPUT_DIR / "sentences_masked.json").write_text(json.dumps(sentences_masked, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Đã lưu checkpoint NLI:", NLI_CHECKPOINT)

# Tái tạo captions che alias khi checkpoint đã tồn tại; không cần chạy model.
if not (OUTPUT_DIR / "sentences_masked.json").is_file():
    sentences_masked = {case["case_id"]: split_sentences(mask_disease_aliases(case["caption"], ALIASES)) for case in CASES}
    (OUTPUT_DIR / "sentences_masked.json").write_text(json.dumps(sentences_masked, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    sentences_masked = json.loads((OUTPUT_DIR / "sentences_masked.json").read_text(encoding="utf-8"))

label_text_nli = nli_arrays["label_text_nli"]
p_entail = nli_arrays["p_entail"]
p_contra = nli_arrays["p_contra"]
best_sentence_idx = nli_arrays["best_sentence_idx"]
llm_group = np.zeros(label_text_nli.shape, dtype=np.int8)
dry_rows = []
negation_terms = tuple(term.casefold() for term in cfg["llm_verify"]["negation_terms"])
for row, case in enumerate(CASES):
    for column, proposition in enumerate(PROPOSITIONS):
        value = label_text_nli[row, column]
        sentence_index = int(best_sentence_idx[row, column])
        evidence = sentences_masked[case["case_id"]][sentence_index].casefold() if sentence_index >= 0 else ""
        group = ""
        if value == PRESENT and proposition.category in {"symptom", "history"} and cfg["llm_verify"].get("group_d", False):
            group = "D"
        elif value == PRESENT and any(term in evidence for term in negation_terms) and cfg["llm_verify"].get("group_a", False):
            group = "A"
        elif value == UNKNOWN and 0.5 <= float(p_contra[row, column]) < float(cfg["text_labeler"]["contradiction_threshold"]) and cfg["llm_verify"].get("group_b", False):
            group = "B"
        elif value == UNKNOWN and 0.5 <= float(p_entail[row, column]) < float(cfg["text_labeler"]["entailment_threshold"]) and cfg["llm_verify"].get("group_c", False):
            group = "C"
        if group:
            llm_group[row, column] = LLM_GROUP_CODES[group]
            dry_rows.append({"group": group, "split": case["split"], "case_id": case["case_id"], "estimated_input_tokens": len(" ".join(sentences_masked[case["case_id"]])) // 4 + sum(len(h) // 4 for h in proposition.sub_hypotheses)})

dry_run = pd.DataFrame(dry_rows)
if dry_run.empty:
    dry_report = pd.DataFrame(columns=["group", "split", "n_cells", "n_cases", "estimated_calls", "estimated_input_tokens"])
else:
    dry_report = dry_run.groupby(["group", "split"], as_index=False).agg(
        n_cells=("case_id", "size"),
        n_cases=("case_id", "nunique"),
        estimated_input_tokens=("estimated_input_tokens", "sum"),
    )
    dry_report["estimated_calls"] = dry_report["n_cases"]
    dry_report = dry_report[["group", "split", "n_cells", "n_cases", "estimated_calls", "estimated_input_tokens"]]
dry_report.to_csv(OUTPUT_DIR / "llm_dry_run_report.csv", index=False)
display(dry_report)
print("NLI checkpoint và LLM dry-run report đã được lưu. Không có API call.")

## 8. Chạy ConceptCLIP độc lập và lưu checkpoint ảnh

Ô này chỉ được chạy sau khi ô NLI báo đã lưu checkpoint. Nó không gọi NLI. Nếu ConceptCLIP lỗi, traceback hiển thị trực tiếp ở đây; checkpoint NLI vẫn nguyên vẹn. Với ConceptCLIP, thực hiện kiểm tra API của model và một ảnh trước khi chạy toàn bộ tập dữ liệu.

In [ ]:
if IMAGE_CHECKPOINT.is_file():
    with np.load(IMAGE_CHECKPOINT, allow_pickle=False) as archive:
        required = {"case_id", "proposition_id", "label_img", "p_img"}
        missing = required - set(archive.files)
        if missing:
            raise ValueError(f"ConceptCLIP checkpoint thiếu arrays: {sorted(missing)}")
        if not np.array_equal(archive["case_id"], nli_arrays["case_id"]):
            raise ValueError("ConceptCLIP checkpoint không khớp NLI checkpoint.")
        if not np.array_equal(archive["proposition_id"], nli_arrays["proposition_id"]):
            raise ValueError("ConceptCLIP checkpoint không khớp propositions WS_02.")
        image_arrays = {key: archive[key] for key in required}
    print("Đã nạp checkpoint ConceptCLIP, không chạy lại:", IMAGE_CHECKPOINT)
else:
    from PIL import Image
    from transformers import AutoModel, AutoProcessor

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Cần GPU Colab trước khi chạy ConceptCLIP toàn bộ.")
    options = cfg["image_labeler"]
    print("Đang tải ConceptCLIP:", options["model_id"])
    clip_model = AutoModel.from_pretrained(options["model_id"], trust_remote_code=True).to(device).eval()
    clip_processor = AutoProcessor.from_pretrained(options["model_id"], trust_remote_code=True)
    if not hasattr(clip_model, "get_text_features") or not hasattr(clip_model, "get_image_features"):
        raise RuntimeError("ConceptCLIP không cung cấp get_text_features/get_image_features tương thích với notebook này.")

    applicable = [(index, prop) for index, prop in enumerate(PROPOSITIONS) if prop.image_applicable]
    prompts = [
        text
        for _, prop in applicable
        for hypothesis in prop.sub_hypotheses
        for text in (
            f"a clinical photo of skin with {hypothesis}",
            f"this skin lesion shows {hypothesis}",
            f"a clinical photo of skin without {hypothesis}",
            f"this skin lesion does not show {hypothesis}",
        )
    ]
    text_inputs = clip_processor(text=prompts, return_tensors="pt", padding=True, truncation=True)
    text_inputs = {key: value.to(device) for key, value in text_inputs.items()}
    with torch.inference_mode():
        text_features = clip_model.get_text_features(**text_inputs)
    text_features = torch.nn.functional.normalize(text_features.float(), dim=-1)

    # Smoke test một ảnh: lỗi model/API/ảnh đầu vào sẽ xuất hiện trước full run.
    first_path = PATHS["image_root"] / CASES[0]["case_id"]
    with Image.open(first_path).convert("RGB") as first_image:
        smoke_inputs = clip_processor(images=first_image, return_tensors="pt")
    smoke_inputs = {key: value.to(device) for key, value in smoke_inputs.items()}
    with torch.inference_mode():
        smoke_feature = clip_model.get_image_features(**smoke_inputs)
    assert smoke_feature.shape[0] == 1, "ConceptCLIP smoke test trả về batch ảnh không hợp lệ."
    print("ConceptCLIP smoke test thành công:", first_path.name)

    label_img = np.full(label_text_nli.shape, UNKNOWN, dtype=np.int8)
    p_img = np.full(label_text_nli.shape, np.nan, dtype=np.float16)
    for row, case in enumerate(tqdm(CASES, desc="WS_03 ConceptCLIP")):
        image_path = PATHS["image_root"] / case["case_id"]
        if not image_path.is_file():
            raise FileNotFoundError(f"Không tìm thấy ảnh: {image_path}")
        with Image.open(image_path).convert("RGB") as image:
            image_inputs = clip_processor(images=image, return_tensors="pt")
        image_inputs = {key: value.to(device) for key, value in image_inputs.items()}
        with torch.inference_mode():
            image_feature = clip_model.get_image_features(**image_inputs)
        image_feature = torch.nn.functional.normalize(image_feature.float(), dim=-1)[0]
        offset = 0  # Reset ở mỗi ảnh: cùng prompt embedding được dùng lại cho mọi case.
        for column, prop in applicable:
            hypothesis_scores = []
            for _ in prop.sub_hypotheses:
                pos = (image_feature @ text_features[offset:offset + 2].T).mean()
                neg = (image_feature @ text_features[offset + 2:offset + 4].T).mean()
                logit_scale = getattr(clip_model, "logit_scale", torch.tensor(1.0, device=device))
                scale = logit_scale.exp() if torch.is_tensor(logit_scale) else 1.0
                hypothesis_scores.append(float(torch.softmax(scale * torch.stack((pos, neg)), dim=0)[0]))
                offset += 4
            score = max(hypothesis_scores)
            p_img[row, column] = score
            if score >= float(options["positive_threshold"]):
                label_img[row, column] = PRESENT
            elif all(value <= float(options["negative_threshold"]) for value in hypothesis_scores):
                label_img[row, column] = ABSENT

    image_arrays = {
        "case_id": nli_arrays["case_id"],
        "proposition_id": nli_arrays["proposition_id"],
        "label_img": label_img,
        "p_img": p_img,
    }
    np.savez_compressed(IMAGE_CHECKPOINT, **image_arrays)
    print("Đã lưu checkpoint ConceptCLIP:", IMAGE_CHECKPOINT)

## 9. Hợp nhất checkpoint và tạo artifact WS_03 cuối cùng

Ô này chỉ làm việc khi cả hai checkpoint tồn tại. Nó không chạy lại NLI hoặc ConceptCLIP. Trong lần chạy này, `label_text` vẫn bằng `label_text_nli`; các nhóm LLM chỉ là dry-run, không có API call.

In [ ]:
if not NLI_CHECKPOINT.is_file() or not IMAGE_CHECKPOINT.is_file():
    raise RuntimeError("Cần cả NLI và ConceptCLIP checkpoint trước khi hợp nhất nhãn.")

with np.load(NLI_CHECKPOINT, allow_pickle=False) as archive:
    nli_arrays = {key: archive[key] for key in archive.files}
with np.load(IMAGE_CHECKPOINT, allow_pickle=False) as archive:
    image_arrays = {key: archive[key] for key in archive.files}

if not np.array_equal(nli_arrays["case_id"], image_arrays["case_id"]):
    raise ValueError("NLI và ConceptCLIP checkpoint có thứ tự case khác nhau.")
if not np.array_equal(nli_arrays["proposition_id"], image_arrays["proposition_id"]):
    raise ValueError("NLI và ConceptCLIP checkpoint có proposition khác nhau.")

sentences_masked = json.loads((OUTPUT_DIR / "sentences_masked.json").read_text(encoding="utf-8"))
label_text_nli = nli_arrays["label_text_nli"]
p_entail = nli_arrays["p_entail"]
p_contra = nli_arrays["p_contra"]
best_sentence_idx = nli_arrays["best_sentence_idx"]
label_img = image_arrays["label_img"]
p_img = image_arrays["p_img"]
llm_group = np.zeros(label_text_nli.shape, dtype=np.int8)
negation_terms = tuple(term.casefold() for term in cfg["llm_verify"]["negation_terms"])
for row, case in enumerate(CASES):
    for column, proposition in enumerate(PROPOSITIONS):
        value = label_text_nli[row, column]
        sentence_index = int(best_sentence_idx[row, column])
        evidence = sentences_masked[case["case_id"]][sentence_index].casefold() if sentence_index >= 0 else ""
        if value == PRESENT and proposition.category in {"symptom", "history"} and cfg["llm_verify"].get("group_d", False):
            llm_group[row, column] = LLM_GROUP_CODES["D"]
        elif value == PRESENT and any(term in evidence for term in negation_terms) and cfg["llm_verify"].get("group_a", False):
            llm_group[row, column] = LLM_GROUP_CODES["A"]
        elif value == UNKNOWN and 0.5 <= float(p_contra[row, column]) < float(cfg["text_labeler"]["contradiction_threshold"]) and cfg["llm_verify"].get("group_b", False):
            llm_group[row, column] = LLM_GROUP_CODES["B"]
        elif value == UNKNOWN and 0.5 <= float(p_entail[row, column]) < float(cfg["text_labeler"]["entailment_threshold"]) and cfg["llm_verify"].get("group_c", False):
            llm_group[row, column] = LLM_GROUP_CODES["C"]

arrays = {
    "case_id": nli_arrays["case_id"],
    "split": nli_arrays["split"],
    "proposition_id": nli_arrays["proposition_id"],
    "label_text_nli": label_text_nli,
    "label_text": label_text_nli.copy(),
    "llm_group": llm_group,
    "llm_verdict": np.full(label_text_nli.shape, LLM_NOT_SENT, dtype=np.int8),
    "llm_quote_invalid": np.zeros(label_text_nli.shape, dtype=np.int8),
    "label_img": label_img,
    "label_final": merge_labels(label_text_nli, label_img, teacher="both"),
    "p_entail": p_entail,
    "p_contra": p_contra,
    "p_img": p_img,
    "best_sentence_idx": best_sentence_idx,
}
labels = WeakLabels(arrays)
labels_path = labels.save(OUTPUT_DIR / "labels.npz")
meta = {
    "stage": "WS_03_weak_labels",
    "teacher": "both",
    "llm_mode": "dry-run",
    "seed": cfg["seed"],
    "n_cases": len(CASES),
    "n_propositions": len(PROPOSITIONS),
    "case_disease": [{"case_id": case["case_id"], "disease_id": case["disease"]} for case in CASES],
    "propositions_final_sha256": sha256_file(PATHS["propositions_final"]),
    "propositions_review_sha256": sha256_file(PATHS["propositions_review"]),
    "teacher_text_model": str(PATHS["teacher_text_model"]),
    "thresholds": {"text": cfg["text_labeler"], "image": cfg["image_labeler"]},
    "checkpoints": {"nli": str(NLI_CHECKPOINT), "conceptclip": str(IMAGE_CHECKPOINT)},
}
(OUTPUT_DIR / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
survey_dir = write_survey(labels, PROPOSITIONS, OUTPUT_DIR / "survey", case_diseases=[case["disease"] for case in CASES])
print("Đã lưu:", labels_path)
print("Đã lưu:", OUTPUT_DIR / "meta.json")
print("Survey:", survey_dir)

## 10. Xem khảo sát nhãn

Các bảng bên dưới chỉ đọc `labels.npz` và survey đã sinh, không thay đổi nhãn.

In [ ]:
SURVEY_DIR = PATHS["weak_labels_dir"] / "survey"
for filename in ["stats_per_proposition.csv", "stats_per_group.csv", "text_image_agreement.csv", "llm_impact.csv"]:
    path = SURVEY_DIR / filename
    print(f"\n--- {filename} ---")
    display(pd.read_csv(path).head(20))

print((SURVEY_DIR / "survey_summary.md").read_text(encoding="utf-8"))